# 0.0 Setup: Getting Ready to Analyze All of Us Data

Welcome! This series of notebooks walks you through a complete research project using **All of Us** data, from pulling data out of the database to running the kinds of analyses you'd see in a clinical research paper.

**This notebook gets you set up.** Run it once before the others.

### By the end of this notebook you'll be able to

1. Run code in a Jupyter notebook and read its output
2. Use the basic building blocks of R: objects, functions, packages, data frames, and the pipe `%>%`
3. Point the notebooks at **your** bucket and the All of Us dataset
4. Check that you can query the All of Us data

| | |
|---|---|
| ⏱️ **Time** | 15–20 minutes |
| 📂 **Uses** | nothing yet |
| 💾 **Saves** | nothing |

**Before you start:**

- An All of Us workspace with the **Controlled Tier** data collection
- A **bucket** in that workspace (Resources tab → **+ New resource** → Cloud Storage bucket)
- This notebook open in an **R** app (e.g., AoU Jupyter with the R kernel)

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## The study we'll work through

**Research question:** how do adults **with** and **without diabetes** differ in their demographics, their care for post-traumatic stress disorder (PTSD), and their cardiometabolic measurements (BMI, blood pressure, cholesterol, HbA1c)? And among people with diabetes, how long until they develop **chronic kidney disease (CKD)**?

Each notebook is one step, and can be taught as its own short class:

| Notebook | What you'll do | Class |
|---|---|---|
| **0.0 setup** | Get ready (this notebook) | Any |
| **1.1 build_cohorts** | Pull the study and control groups out of the database with SQL | 1 |
| **1.2 survival_data** | Pull diagnosis and visit dates for the survival analysis | 1 or 5 |
| **2.0 clean_and_join** | Clean the raw data and combine it into one analysis table | 2 |
| **3.1 table_one** | Describe and compare the two groups | 3 |
| **3.2 logistic_regression** | Which factors are linked to the **odds** of diabetes? | 4 |
| **3.3 linear_regression** | Which factors are linked to **blood pressure**? | 4 |
| **3.4 survival_analysis** | How long until people with diabetes develop CKD? | 5 |

Each notebook saves its results to your bucket, and the next one reads them. So once the data is saved, a class can start at any notebook.

## Part 1: Jupyter and R in 5 minutes

*Already comfortable with R and Jupyter? Skip to Part 2.*

### Running code

A notebook is made of **cells**:

- **Text cells** (like this one) explain what's going on
- **Code cells** (grey boxes) hold R code

To run a code cell, click it and press **Shift + Enter**. While it runs, you'll see `[*]` to its left; when it's done, a number appears, like `[3]`, and any output shows up underneath.

> 💡 Run cells **in order**, top to bottom. Each cell can use things created by the cells above it.

Try it: run the cell below.

In [ ]:
1 + 1

### Objects: giving things a name

`<-` (read it as "gets") saves a value under a name, so you can use it later. Nothing is printed when you save something; type the name on its own to see it.

In [ ]:
ages <- c(34, 41, 29, 50, 38)   # c() combines values into a vector
ages

### Functions: doing things

A **function** takes inputs (in parentheses) and returns a result. R has thousands of them.

In [ ]:
mean(ages)     # the average
length(ages)   # how many values
round(sd(ages), 1)   # the standard deviation, rounded to 1 decimal place

### Packages: borrowing other people's functions

A **package** is a bundle of functions someone has written and shared. You load one with `library()`. We mostly use the **tidyverse**, a family of packages for working with data (`dplyr`, `tidyr`, `readr`, `stringr`, `ggplot2`…).

In [ ]:
library(tidyverse)

### Data frames: tables of data

A **data frame** (the tidyverse calls it a **tibble**) is a table: each **row** is one observation (here, one person) and each **column** is one variable.

In [ ]:
people <- tibble(
  person   = c("A", "B", "C", "D", "E"),
  age      = ages,
  diabetes = c("No", "Yes", "No", "Yes", "No")
)
people

### The pipe `%>%`: one step after another

The **pipe** `%>%` passes what's on its left into the function on its right. Read it as **"and then"**. It lets you write a series of steps in the order you do them:

In [ ]:
people %>%                    # take the people table, and then
  filter(diabetes == "Yes") %>%   # keep only rows where diabetes is "Yes", and then
  summarise(mean_age = mean(age)) # calculate their average age

That's most of what you need. Each notebook explains new functions as they come up, with a **📦 Functions used** table.

### ✍️ Your turn

Using the `people` table, find the average age of people **without** diabetes.

<details>
<summary><b>Show answer</b></summary>

```r
people %>%
  filter(diabetes == "No") %>%
  summarise(mean_age = mean(age))
```

It's the same code as above with `"Yes"` changed to `"No"`. The answer is 33.7.

</details>

In [ ]:
# Your code here

## Part 2: Connect to the All of Us data

### How the pieces fit together

- **The data** lives in **BigQuery**, Google's cloud database, as a set of tables called the **Curated Data Repository (CDR)**. We get data out of it by writing **SQL** queries.
- **Your bucket** is cloud storage that belongs to your workspace. It shows up in this app as a **folder**, so we can save and read files there like on a normal computer. Files in your bucket stay there even if you delete this app.
- **`aou_setup.R`** is a small file in the same folder as these notebooks. Every notebook starts by running it with `source("aou_setup.R")`. It:
  - loads the packages we need (and installs any that are missing, the first time)
  - holds **two settings** you edit once
  - creates **helper functions** used in every notebook:

| Helper | What it does |
|---|---|
| `run_query(sql)` | Sends a SQL query to BigQuery and returns the result as a data frame |
| `save_data(df, "file.csv")` | Saves a data frame as a CSV file in your bucket's `data/` folder |
| `load_data("file.csv")` | Reads a CSV file from your bucket's `data/` folder |

### Step 1: Find your bucket's folder name

Your buckets appear as folders under `~/workspace` (`~` means your home folder). This lists them:

In [ ]:
list.dirs(file.path(Sys.getenv("HOME"), "workspace"), recursive = FALSE)

**🔍 Read the output:** you'll see one path per bucket. The part after the last `/` is the **folder name**. You may also see folders for the All of Us data collection (e.g. `vwb-aou-datasets-…`) or for a Spark app (`dataproc-…`); those aren't yours to save into.

**No folders?** Create a bucket on the **Resources** tab, then restart this app (or run `wb resource mount` in a terminal).

### Step 2: Edit the two settings in `aou_setup.R`

Open `aou_setup.R` (double-click it in the file browser on the left) and change:

- **`BUCKET_NAME`:** your bucket's folder name from Step 1, e.g. `"workspace-bucket"`
- **`CURRENT_DATASET`:** which version of the All of Us data to use. Find it in any **Data Explorer** query in your workspace: it's the `project.dataset` part of the `FROM` line, e.g. `wb-silky-artichoke-2408.C2025Q4R6` (Controlled Tier CDR v9).

Save the file (**Ctrl/Cmd + S**), then close its tab.

### Step 3: Load the settings

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

**🔍 Read the output:** you should see *"Setup done. Dataset: … | Data folder: …"*. The first time, you may also see *"Installing missing packages…"*: that's normal and takes a few minutes.

### Step 4: Test the connection

This query counts everyone in the dataset. It's tiny and costs almost nothing.

In [ ]:
run_query(str_glue("SELECT COUNT(*) AS participants FROM `{CURRENT_DATASET}.person`"))

**🔍 Read the output:** a single number in the **hundreds of thousands**. That's how many people are in this version of the All of Us dataset.

`str_glue()` (from the **stringr** package) builds text by filling in anything inside `{ }`: here it swaps `{CURRENT_DATASET}` for your dataset path before the query is sent.

> 💰 **About cost:** every query is billed to your workspace's billing account, based on how much data it reads. This one is free or close to it. The queries in 1.1 and 1.2 cost more, so we run them **once** and save the results.

### ✍️ Your turn

Count how many rows are in the `measurement` table. (Hint: copy the query above and change one word.)

<details>
<summary><b>Show answer</b></summary>

```r
run_query(str_glue("SELECT COUNT(*) AS n FROM `{CURRENT_DATASET}.measurement`"))
```

You'll see a number in the **hundreds of millions**: every lab result and vital sign for every participant. That's why we always filter queries down to just the people and measurements we need.

</details>

In [ ]:
# Your code here

## ⚠️ Common errors

| Error message | What it means | Fix |
|---|---|---|
| *Can't find your bucket folder* | `BUCKET_NAME` doesn't match a folder | Copy the folder name from Step 1 into `aou_setup.R`, save, rerun the setup cell |
| *could not find function "str_glue"* (or `run_query`) | The setup cell hasn't run | Run `source("aou_setup.R")` first |
| *Access Denied* or *Not found: Dataset* | `CURRENT_DATASET` is wrong, or you don't have access to that tier | Copy it again from a Data Explorer query in **this** workspace |
| The cell shows `[*]` for a long time | It's still running | Wait. Don't run it again, or it starts over |

## ✅ Key takeaways

- Run cells **in order** with **Shift + Enter**
- Every notebook starts with `source("aou_setup.R")`
- The data is in **BigQuery**; we query it with **SQL** and save results to our **bucket**
- Queries cost money: run them once, save the results

## Where next?

- **Starting fresh:** go to **1.1 build_cohorts**.
- **Already have CSV files from the earlier (classic Workbench) versions of these notebooks?** They use the same file names, so you can **skip 1.1 and 1.2** and go straight to **2.0 clean_and_join**. Just make sure `BUCKET_NAME` points at the bucket that holds them.